# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
SPARSE_PATHS = [
    'README.md', 'docs', 'src', 'scripts', 'config', 'colab_notebooks', 'requirements.txt', 'requirements_colab.txt', 'pyproject.toml',
]
REQUIRED_BOOTSTRAP_FILES = ['requirements_colab.txt']

def _repair_sparse_checkout(repo_root):
    missing = [path for path in REQUIRED_BOOTSTRAP_FILES if not (repo_root / path).exists()]
    if missing and (repo_root / '.git').exists():
        print(f'[BOOTSTRAP] Adding missing sparse checkout paths: {missing}', flush=True)
        subprocess.run(['git', 'sparse-checkout', 'add', *missing], cwd=str(repo_root), check=True)

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            _repair_sparse_checkout(repo_root)
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if CLONE_TARGET.exists():
        shutil.rmtree(CLONE_TARGET)
    print(f'[BOOTSTRAP] Sparse cloning training surface to {CLONE_TARGET}...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, str(CLONE_TARGET)], check=True)
    print('[BOOTSTRAP] Selecting source checkout paths...', flush=True)
    subprocess.run(['git', 'sparse-checkout', 'set', *SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


[BOOTSTRAP] Sparse cloning training surface to /content/bitirmeprojesi...
[BOOTSTRAP] Selecting source checkout paths...
[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SETUP] Checking model access for training...
[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.
[KONTROL] Varsayilan backbone modeli: facebook/dinov3-vitl16-pretrain-lvd1689m


In [2]:
# Notebook 2 calisma kimligi
# Once bu hucreyi duzenleyin, sonra bootstrap hucresini yeniden calistirin.
CROP_NAME = "tomato"
PART_NAME = "unspecified"
ENABLE_BAYESIAN_OPTIMIZATION = False  # Bayesian optimization devre disi
print(f"[RUN] crop={CROP_NAME} part={PART_NAME} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}")


[RUN] crop=tomato part=unspecified bayes_opt=False


In [3]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())

[BOOTSTRAP] run_id=tomato_unspecified_2026-05-05_16-31-22 crop=tomato part=unspecified


In [4]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri ---
    # Bu hucreyi duzenleyin, sonra kalan hucreleri sirayla calistirin.
    # Kosu kimligi icin CROP_NAME/PART_NAME degerlerini ustteki hucreden yonetin.

    # RUNTIME_DATASET_ROOT: Notebook 0'un yazdigi <dataset_key>/continual|val|test|ood yapisini tutan repo-ici root.
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"

    # DATASET_NAME: Notebook 0'un urettigi runtime dataset klasor adi. Bos ise notebook kullaniciya sorar.
    DATASET_NAME = ""

    # OOD_ROOT: Gercek OOD klasoru. Bos ise ASK_FOR_OOD_ROOT=True iken notebook yol sorar; Enter varsa runtime ood/ kullanir.
    OOD_ROOT = ""
    ASK_FOR_OOD_ROOT = True

    # OE_ROOT: Outlier Exposure egitim klasoru. OE_ENABLED=True ve bos ise ASK_FOR_OE_ROOT=True iken notebook yol sorar; Enter varsa runtime oe/ kullanir.
    OE_ROOT = ""
    ASK_FOR_OE_ROOT = True
    OE_ENABLED = False
    OE_LOSS_WEIGHT = 0.5

    # CROP_NAME ve PART_NAME, kosu adlandirmasi ve metadata icin kullanilir.
    CROP_NAME = globals().get("CROP_NAME", "tomato")
    PART_NAME = globals().get("PART_NAME", "unspecified")
    ENABLE_BAYESIAN_OPTIMIZATION = bool(globals().get("ENABLE_BAYESIAN_OPTIMIZATION", False))

    # ALLOW_UNDER_MIN_TRAINING: True olursa 100 image/class production guardrail'i research kosulari icin bypass edilir.
    ALLOW_UNDER_MIN_TRAINING = False

    # EPOCHS: train split uzerinden kac tam gecis yapilacagi.
    EPOCHS = 12

    # BATCH_SIZE: optimizer adimi basina ornek sayisi. GPU limitine yakin olacak sekilde artirilabilir.
    BATCH_SIZE = 96

    # LEARNING_RATE: adapter/LoRA parametreleri icin optimizer adim buyuklugu.
    LEARNING_RATE = 2e-4

    # LORA_R: LoRA rank degeri. Buyudukce kapasite ve VRAM/islem maliyeti artar.
    LORA_R = 24

    # LORA_ALPHA: LoRA olcekleme katsayisi. Genelde LORA_R degerinin 2x-4x araliginda kullanilir.
    LORA_ALPHA = 24

    # LORA_DROPOUT: LoRA katmanlarina uygulanan dropout. Buyudukce regularizasyon artar.
    LORA_DROPOUT = 0.1

    # OOD_FACTOR: OOD esik hassasiyetini carpansal olarak ayarlar.
    OOD_FACTOR = 3.0
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: eski/yeni sinif enerji ayrimi icin deneysel egitim regularizeri.
    BER_ENABLED = False

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: eski ve yeni sinif kisimlari icin BER ceza agirliklari.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW weight decay degeri.
    WEIGHT_DECAY = 0.01

    # MIXED_PRECISION: {'off', 'auto', 'fp16', 'bf16'} seceneklerinden biri.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation katsayisi.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping esigi. 0 olursa clipping kapanir.
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing katsayisi.
    LABEL_SMOOTHING = 0.0

    # LOSS_NAME / LOGITNORM_TAU: varsayilan kayip LogitNorm'dur; CE icin loss_name='cross_entropy' secin.
    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    # Scheduler ayarlari.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping ayarlari.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    # Tekrarlanabilirlik ve runtime ayarlari.
    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    # NUM_WORKERS: dataloader worker sayisi. CPU veri yukleme paralelligini belirler.
    NUM_WORKERS = 12

    # PREFETCH: NUM_WORKERS > 0 iken worker basina prefetch katsayisi.
    PREFETCH = 8

    # PIN_MEMORY: host memory sabitleyerek host-to-GPU aktarimini hizlandirir.
    PIN_MEMORY = True

    # USE_CACHE: destekleniyorsa decode edilmis ornekleri RAM'de tutar.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: continual/train split'ini de cache'ler. Yuksek RAM'li Colab icin uygundur.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: her N epoch'ta tam validation calistirir; final epoch her zaman dahildir.
    VALIDATION_EVERY_N_EPOCHS = 2

    # CHECKPOINT_EVERY_N_STEPS / CHECKPOINT_ON_EXCEPTION: notebook checkpoint sikligi ayarlari.
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True

    # STDOUT_BATCH_INTERVAL: canli training ilerleme yazdirma araligi.
    STDOUT_BATCH_INTERVAL = 12

    # RESUME_MODE: "fresh" yeni kosu baslatir, "resume" son checkpointten devam eder.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # AUTO_DISCONNECT_RUNTIME: tum final exportlar basariliysa Colab runtime'i kapatir.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: son durum gorunsun diye disconnect oncesi kisa bekleme suresi.
    AUTO_DISCONNECT_GRACE_SECONDS = 20

    # AUTO_PUSH_TO_GITHUB: final exportlar bitince runs/<crop>/<part>/<RUN_ID>/ klasorunu repoya commit edip pushlar.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: auto-push aciksa kullanilacak git remote adi.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: auto-push icin branch override degeri. None olursa mevcut branch kullanilir.
    AUTO_PUSH_BRANCH = None

    # MANUAL_PARAM_OVERRIDES: anahtarlar notebook degisken adlariyla birebir ayni olmali.
    # Ornek: {"BATCH_SIZE": 32, "EPOCHS": 16, "PIN_MEMORY": False}
    MANUAL_PARAM_OVERRIDES = {}
ADAPTER_KEY = "grape__leaf"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf

RECS = {
    "grape__fruit": {
        "crop": "grape", "part": "fruit",
        "ood": "data/prepared_runtime_datasets/grape__fruit/ood",
        "oe": "data/prepared_runtime_datasets/grape__fruit/oe",
        "oe_enabled": True, "oe_w": 0.20,
        "allow_under_min": False,
        "overrides": {"EPOCHS": 32, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0},
    },
    "grape__leaf": {
        "crop": "grape", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/grape__leaf/ood",
        "oe": "data/prepared_runtime_datasets/grape__leaf/oe",
        "oe_enabled": True, "oe_w": 0.20,
        "allow_under_min": False,
        "overrides": {"EPOCHS": 28, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.0},
    },
    "strawberry__fruit": {
        "crop": "strawberry", "part": "fruit",
        "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood",
        "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe",
        "oe_enabled": True, "oe_w": 0.10,
        "allow_under_min": True,
        "overrides": {"EPOCHS": 36, "BATCH_SIZE": 32, "LEARNING_RATE": 8e-5, "LORA_R": 16, "LORA_ALPHA": 16, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 3.0},
    },
    "strawberry__leaf": {
        "crop": "strawberry", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood",
        "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe",
        "oe_enabled": True, "oe_w": 0.15,
        "allow_under_min": False,
        "overrides": {"EPOCHS": 22, "BATCH_SIZE": 96, "LEARNING_RATE": 1.5e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0},
    },
   "tomato__fruit": {
    "crop": "tomato", "part": "fruit",
    "ood": "data/ood_dataset/final/tomato__fruit_ood_final",
    "oe": "data/prepared_runtime_datasets/tomato__fruit/oe",
    "oe_enabled": True, "oe_w": 0.15,
    "allow_under_min": False,
    "overrides": {
        "EPOCHS": 30,
        "BATCH_SIZE": 64,
        "LEARNING_RATE": 8e-5,
        "LORA_R": 24,
        "LORA_ALPHA": 24,
        "LORA_DROPOUT": 0.15,
        "OOD_FACTOR": 3.0,
    },
},

    "tomato__leaf": {
        "crop": "tomato", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/tomato__leaf/ood",
        "oe": "data/prepared_runtime_datasets/tomato__leaf/oe",
        "oe_enabled": True, "oe_w": 0.15,
        "allow_under_min": False,
        "overrides": {"EPOCHS": 20, "BATCH_SIZE": 96, "LEARNING_RATE": 1.2e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0},
    },
}

rec = RECS[ADAPTER_KEY]

CROP_NAME = rec["crop"]
PART_NAME = rec["part"]
DATASET_NAME = ADAPTER_KEY
RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"

OOD_ROOT = rec["ood"]
ASK_FOR_OOD_ROOT = False

OE_ROOT = rec["oe"]
OE_ENABLED = rec["oe_enabled"]
ASK_FOR_OE_ROOT = False
OE_LOSS_WEIGHT = rec["oe_w"]

ALLOW_UNDER_MIN_TRAINING = rec["allow_under_min"]
VALIDATION_EVERY_N_EPOCHS = 1

MANUAL_PARAM_OVERRIDES = {
    **rec["overrides"],
    "WEIGHT_DECAY": 0.01,
    "LOSS_NAME": "logitnorm",
    "LOGITNORM_TAU": 1.0,
    "MIXED_PRECISION": "bf16",
    "GRAD_ACCUM_STEPS": 1,
    "VALIDATION_EVERY_N_EPOCHS": 1,
    "EARLY_STOPPING_PATIENCE": 6,
    "RANDAUGMENT_NUM_OPS": 2,
    "RANDAUGMENT_MAGNITUDE": 7,
    "NUM_WORKERS": 12,
    "PREFETCH": 8,
    "CACHE_TRAIN_SPLIT": True,
}

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell04_parameter_resolution.py', globals())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=grape epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=False
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=grape__leaf
[OOD] ood_root=data/prepared_runtime_datasets/grape__leaf/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/grape__leaf/oe ask_for_oe_root=False enabled=True loss_weight=0.2
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [5]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [6]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


[BOOTSTRAP] Fetching selected Notebook 2 dataset paths: ['data/prepared_runtime_datasets/grape__leaf', 'data/ood_dataset/final/grape__leaf_ood_final', 'data/oe_dataset/grape_leaf_oe_unsupported_leaf_candidates', 'data/prepared_runtime_datasets/grape__leaf/ood', 'data/prepared_runtime_datasets/grape__leaf/oe']
[OOD] explicit ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/ood
[OE] explicit oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/oe
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf classes=7: ['üzüm_antraknoz_yaprak', 'üzüm_kav_esca_yaprak', 'üzüm_külleme_yaprak', 'üzüm_leafroll_virüs_yaprak', 'üzüm_mildiyö_yaprak', 'üzüm_sağlıklı_yaprak', 'üzüm_yelpaze_virüsü_yaprak']
[DATASET][CHECK] scale=small continual=1206 val=324 test=324 ood=249 classes=7
[DATASET][WARN] Manifest rows include 174 eval-quality-risk item(s); prefer cautious regularization.
[PARAMS] Manual overr

In [7]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


[OOD] selected ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/ood
[OE] selected oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/oe
[CLASSES] ['u_zu_m_antraknoz_yaprak', 'u_zu_m_kav_esca_yaprak', 'u_zu_m_ku_lleme_yaprak', 'u_zu_m_leafroll_viru_s_yaprak', 'u_zu_m_mildiyo_yaprak', 'u_zu_m_sag_lıklı_yaprak', 'u_zu_m_yelpaze_viru_su_yaprak']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=0 unmatched=7
[CLASSES] taxonomy-unmatched classes kept: ['u_zu_m_antraknoz_yaprak', 'u_zu_m_kav_esca_yaprak', 'u_zu_m_ku_lleme_yaprak', 'u_zu_m_leafroll_viru_s_yaprak', 'u_zu_m_mildiyo_yaprak', 'u_zu_m_sag_lıklı_yaprak', 'u_zu_m_yelpaze_viru_su_yaprak']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=13,772,555  classes=7


In [8]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [9]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


[TRAIN] epochs=28 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/18 loss=0.0000 lr=0.000100 speed=255.7s/s elapsed=7s eta=288s
[CKPT] epoch_end epoch=1 step=18
[EPOCH] 1/28: train_loss=1.9899 val_loss=1.9318 val_acc=0.1636 macro_f1=0.0403 * BEST
[TRAIN] epoch=2 batch=12/18 loss=0.0000 lr=0.000099 speed=255.1s/s elapsed=19s eta=307s
[EPOCH] 2/28: train_loss=1.9430 val_loss=1.9388 val_acc=0.0895 macro_f1=0.0534
[TRAIN] epoch=3 batch=12/18 loss=0.0000 lr=0.000098 speed=255.1s/s elapsed=27s eta=254s
[EPOCH] 3/28: train_loss=1.9233 val_loss=1.9687 val_acc=0.1173 macro_f1=0.0798
[TRAIN] epoch=4 batch=12/18 loss=0.0000 lr=0.000096 speed=255.3s/s elapsed=34s eta=225s
[CKPT] epoch_end epoch=4 step=72
[EPOCH] 4/28: train_loss=1.8352 val_loss=1.5356 val_acc=0.4136 macro_f1=0.3631 * BEST
[TRAIN] epoch=5 batch=12/18 loss=0.0000 lr=0.000093 speed=255.7s/s elapsed=44s eta=221s
[CKPT] epoch_end epoch=5 step=90
[EPOCH] 5/28: train_loss=1.6745 val_loss=1.3863 val_acc=0.4599 macro_f1=0.3921 * BES

In [10]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


[OOD] Kalibrasyon tamamlandi. classes=7 version=1


In [11]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/tomato_unspecified_2026-05-05_16-31-22/artifacts/adapter_export/grape/leaf/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())


[DOGRULAMA (referans)] ornek=324 sinif=7 accuracy=0.8241 ood_auroc=0.8034 sure_ds_f1=0.2770 conformal_cov=0.9537
[TEST (ayrilmis)] ornek=324 sinif=7 accuracy=0.8673 ood_auroc=0.8030 sure_ds_f1=0.2842 conformal_cov=0.9599
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
